## ***Parte 2 — app/database.py***

In [ ]:
import sqlalchemy
import databases

DATABASE_URL = "sqlite:///./bank.db"

database = databases.Database(DATABASE_URL)
metadata = sqlalchemy.MetaData()
engine  = sqlalchemy.create_engine(DATABASE_URL, connect_args={"check"})

## ***Parte 3 — models/transaction.py***

In [ ]:
import sqlalchemy
from app.database import metadata

accounts = sqlalchemy.Table(
    "accounts",
    metadata,
    sqlalchemy.Column("id", sqlalchemy.Integer, primary_key= True),
    sqlalchemy.Column("owner", sqlalchemy.String(100), nullable= False),
    sqlalchemy.Column("balance", sqlalchemy.Float, default= 0.0, nullable= False),

)

transactions = sqlalchemy.Table(
    "transactions",
    metadata,
    sqlalchemy.Column("id", sqlalchemy.Integer, primary_key= True),
    sqlalchemy.Column("account_id", sqlalchemy.Integer, sqlalchemy.ForeignKey("accounts.id"), nullable= False),
    sqlalchemy.Column("type", sqlalchemy.String(10), nullable= False), # se será deposit ou withdraw
    sqlalchemy.Column("amount", sqlalchemy.Float, nullable= False),
    sqlalchemy.Column("created_at", sqlalchemy.DateTime, nullable= False),
)

## ***Parte 4 — schemas/transaction.py***

In [ ]:
from datetime import datetime
from pydantic import BaseModel

class TransactionIn(BaseModel):
    account_id: int
    type: str  # ou será deposit ou withdraw
    amount: float

class TransactionUpdateIn(BaseModel):
    type: str | None = None
    amount: float | None = None

## ***Parte 5 — schemas/auth.py***

In [ ]:
from pydantic import BaseModel

class LoginIn(BaseModel):
    user_id: int

## ***Parte 6 — views/transaction.py***

In [ ]:
from datetime import datetime
from pydantic import BaseModel

class TransactionOut(BaseModel):
    id: int
    account_id: int
    type: str
    amount: float
    created_at: datetime

## ***Parte 7 — views/auth.py***

In [ ]:
from pydantic import BaseModel

class LoginOut(BaseModel):
    access_token: str

## ***Parte 8 — services/transaction.py***

In [ ]:
from datetime import datetime

from databases.interfaces import Record
from fastapi import HTTPException, status

from app.database import database
from models.transaction import accounts, transactions
from schemas.transaction import TransactionIn


class TransactionService:
    async def get_balance(self, account_id: int) -> Record:
        return await self.__get_account_by_id(account_id)
    
    async def deposit(self, data: TransactionIn) -> Record:
        if data.amount <= 0:
            raise HTTPException(
                status_code= status.HTTP_400_BAD_REQUEST,
                detail = "Deposit amount mus be positive."
            )
        
    
        accounts = await self.__get_account_by_id(data.account_id)

        # Atualização do saldo
        new_balance = accounts.balance + data.amount
        await database.execute(
            accounts.update()
            .where(accounts.c.id == data.account_id)
            .values(balance = new_balance)

        )

        # Registra transação
        await database.execute(
            transactions.insert().values(
                accounts_id = data.account_id,
                type = "deposit",
                amount = data.amount,
                created_at = datetime.now(),
            )
        )

        return await self.__get_account_by_id(data.account_id)
    
    async def withdraw(self, data: TransactionIn) -> Record:
        if data.amount <= 0:
            raise HTTPException(
                status_code= status.HTTP_400_BAD_REQUEST,
                detail= "Widraw amount must be positive."
            )
        
        accounts = await self.__get_account_by_id(data.account_id)

        if accounts.balance < data.amount:
            raise HTTPException(
                status_code= status.HTTP_400_BAD_REQUEST,
                detail= "Insufficient balance."
            )
        

        # Atualiza saldo
        new_balance = accounts.balance - data.amount
        await database.execute(
            accounts.update()
            .where(accounts.c.id == data.account_id)
            .values(balance = new_balance)
        )

        # Registra transação
        await database.execute(
            transactions.inser().values(
                accounts_id = data.account_id,
                type = "withdraw",
                amount = data.amount,
                created_at = datetime.now(),

            )
        )

        return await self.__get_account_by_id(data.account_id)
    
    async def get_statement(self, account_id: int) -> list[Record]:
        await self.__get_account_by_id(account_id)
        query = transactions.select().where(transactions.c.account_id == account_id)
        return await database.fetch_all(query)
    
    async def __get_account_by_id(self, account_id: int) -> Record:
        query = accounts.select().where(accounts.c.id == account_id)
        result = await database.fetch_one(query)
        if not result:
            raise HTTPException(
                status_code= status.HTTP_404_NOT_FOUND,
                detail= "Account not found."
            )
        return result

## ***Parte 9 — controllers/auth.py***

In [ ]:
from fastapi import APIRouter

from core.security import sign_jwt
from schemas.auth import LoginIn
from views.auth import LoginOut

router = APIRouter(prefix="/auth", tags=["Auth"])

@router.post("/login", response_model= LoginOut)
async def login(data: LoginIn):
    return sign_jwt(user_id = data.user_id)

## ***Parte 10 — controllers/transaction.py***

In [ ]:
from typing import Annotated

from fastapi import APIRouter, Depends, status

from core.security import login_required
from schemas.transaction import TransactionIn
from services.transaction import TransactionService
from views.transaction import TransactionOut, AccountOut


router = APIRouter(prefix="/transactions", tags=["Transactions"])

service = TransactionService()


@router.get("/balance/{account_id}", response_model=AccountOut)
async def get_balance(
    account_id: int,
    current_user: Annotated[dict, Depends(login_required)]
):
    return await service.get_balance(account_id)


@router.post("/deposit", status_code=status.HTTP_201_CREATED, response_model=AccountOut)
async def deposit(
    data: TransactionIn,
    current_user: Annotated[dict, Depends(login_required)]
):
    return await service.deposit(data)


@router.post("/withdraw", status_code=status.HTTP_201_CREATED, response_model=AccountOut)
async def withdraw(
    data: TransactionIn,
    current_user: Annotated[dict, Depends(login_required)]
):
    return await service.withdraw(data)


@router.get("/statement/{account_id}", response_model=list[TransactionOut])
async def get_statement(
    account_id: int,
    current_user: Annotated[dict, Depends(login_required)]
):
    return await service.get_statement(account_id)

## Correção em views/transaction.py

In [ ]:
from datetime import datetime
from pydantic import BaseModel

class TransactionOut(BaseModel):
    id: int
    account_id: int
    type: str
    amount: float
    created_at: datetime

# Adição de código
class AccountOut(BaseModel):
    id: int
    owner: str
    balance: float